# Validation — Étape 3

Ancrage : Gayon, Talay-Değirmenci, Karaesmen & Örmeci (2009), *Optimal Pricing and
Production Policies of a Make-to-Stock System with Fluctuating Demand*, **PEIS** 23(2).

Ce notebook a deux parties, dont le **statut épistémique diffère** et qu'il ne faut pas
confondre :

| | portée | ce qui est établi |
|---|---|---|
| **3.1** | `N=2`, `K=0`, lost sales | reproduction de valeurs **publiées** (Tables 3-5, Figure 2) |
| **3.2** | `N>2`, `K>0`, backlog | **cohérence interne** : cas limites et propriétés structurelles |

Le papier ne couvre que le cas de la première ligne. Les extensions n'ont aucune
référence à reproduire : leur validation est donc plus faible, et le rapport final doit
le dire explicitement plutôt que de laisser croire à une vérification homogène.

Paramètres de référence (§5.3 du papier) : `Λ̄ = 1`, `a = 1`, `c = 0`, `h = 0.01`,
`q = 0.01`, `µ = 0.11`, grille de prix au pas 0.01 (Appendix E).

In [ ]:
import json, sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
import validation as V

# Les résultats sont précalculés par `python -m validation` (quelques minutes).
if V.RESULTS_PATH.exists():
    results = json.loads(V.RESULTS_PATH.read_text(encoding="utf-8"))
    print(f"Résultats chargés depuis {V.RESULTS_PATH.name}")
else:
    results = V.run_all()
    print("Résultats recalculés")

pd.set_option("display.width", 120)

## 3.1 — Validation externe

### Table 4 — niveaux de base-stock

Les base-stocks sont des **entiers** : contrairement aux profit gains, l'exactitude y est
atteignable, et c'est donc le test le plus discriminant des trois tables. Un écart d'une
seule unité signalerait une erreur de modélisation.

In [ ]:
t4 = pd.DataFrame(results["table_4"])
display(t4)
print(f"{t4['exact'].sum()} / {len(t4)} base-stocks reproduits exactement")

### Table 3 — profit gains

Le papier ne publie qu'une décimale, donc son propre arrondi vaut déjà ±0.05 point. La
tolérance retenue est de 0.3 point de pourcentage.

In [ ]:
t3 = pd.DataFrame(results["table_3"])
display(t3)
print(f"écart maximum : {t3['gap_pp'].abs().max():.2f} point de %")

**Lecture.** Le biais est systématiquement positif et de l'ordre de l'arrondi de la
source. L'enseignement de fond est reproduit : les profit gains croissent avec la
variabilité `ε`, et `PG^{edp,s}` reste très proche de `PG^{dp,s}` — c'est-à-dire que
**la tarification dépendante de l'environnement capte l'essentiel du bénéfice de la
tarification dynamique**, pour un coût opérationnel bien moindre.

### Table 5 — prix optimaux

Pour DP, l'intervalle `(p_e, p̄_e)` est pris sur les **états récurrents**
`0 ≤ x ≤ max_e s_e`, comme le prescrit la légende de la Table 5. Le prix en `x = 0` est
exclu : la demande y est perdue, le prix n'a donc aucun effet.

In [ ]:
t5 = pd.DataFrame(results["table_5"])
display(t5)
outliers = t5[t5["gap"] > V.PRICE_TOLERANCE]
print("valeurs hors tolérance :")
display(outliers)

**Deux écarts subsistent, tous deux sur DP.**

| | papier | solveur |
|---|---|---|
| `ε=0.6`, `p^dp_L` (min) | 0.33 | 0.31 |
| `ε=0.8`, `p̄^dp_H` (max) | 0.99 | 0.88 |

La seconde est suspecte du côté du papier. La suite publiée des `p̄^dp_H` est
`0.85 → 0.87 → 0.88 → 0.99` : le dernier terme rompt une progression que le solveur rend
régulière (`0.85 → 0.87 → 0.88 → 0.88`), alors que les quatre autres valeurs DP à
`ε = 0.8` coïncident au centime près. Les résultats sont insensibles à la troncature
(`m` de 60 à 400) et à la tolérance (`1e-5` à `1e-11`).

Ces deux écarts sont **figés dans la suite de tests** : si une évolution du solveur les
faisait disparaître ou en créait d'autres, le test le signalerait au lieu de les laisser
se fondre dans le décor.

### Figure 2 — profit gains en fonction du taux de service

À `ε = 0.8`. La figure ne fait intervenir ni S ni SB.

In [ ]:
fig2 = pd.DataFrame(results["figure_2"])
fig, ax = plt.subplots(figsize=(8, 4.5))
for col, label in [("PG_dp_sp", "$PG^{dp,sp}$"),
                   ("PG_edp_sp", "$PG^{edp,sp}$"),
                   ("PG_dp_edp", "$PG^{dp,edp}$")]:
    ax.plot(fig2["mu"], fig2[col], marker="o", ms=3, label=label)
ax.set_xlabel("taux de service $\\mu$")
ax.set_ylabel("profit gain (%)")
ax.set_title("Profit gains en fonction de $\\mu$  ($\\varepsilon = 0.8$)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

**Lecture.** Trois traits de la Figure 2 du papier sont reproduits :

1. les gains sont **non monotones** en `µ` — ils culminent à capacité intermédiaire.
   À capacité très faible le stock moyen est quasi nul et l'on vend au prix fort en
   permanence ; à capacité abondante la politique de réapprovisionnement suffit à
   absorber les fluctuations. La tarification ne vaut que dans l'entre-deux ;
2. `PG^{dp,sp}` dépasse nettement `PG^{edp,sp}`, mais l'écart entre les deux — c'est-à-dire
   `PG^{dp,edp}` — reste **marginal** (< 3.3 % sur tout le balayage, §5.3.1) ;
3. au-delà de `µ ≈ 0.6`, tous les gains retombent sous 4 %.

C'est l'argument central du papier : le gain vient de l'**adaptation à l'environnement**,
pas de l'adaptation au niveau de stock.

### Propriétés structurelles

Reproduire des chiffres ne suffit pas : les théorèmes du papier prédisent des propriétés
**qualitatives** que le solveur doit exhiber. Elles sont plus robustes que les valeurs
numériques, et un solveur juste par accident les violerait.

In [ ]:
st = pd.DataFrame(results["structural"])
display(st)
assert st.drop(columns="eps").all().all(), "propriété structurelle en défaut"
print("Lemme 1 (concavité), Théorème 2 (prix décroissants en stock),")
print("Théorème 3 (base-stocks ordonnés) et §5.3.2 : vérifiés sur les 4 instances.")

## 3.2 — Cohérence interne des extensions

**Aucune de ces trois extensions n'a de référence publiée à reproduire.** Ce qui suit
atteste que le solveur est cohérent avec lui-même, pas que le modèle est correct au
regard d'une source externe. C'est une garantie réellement plus faible.

| extension | cas limite | ce qu'il établit |
|---|---|---|
| `N > 2` | dédoubler l'environnement H dans un modèle à 3 états | le cas `N=2` est bien un cas particulier |
| `K > 0` | `K = 0` | le modèle à mise en route redonne **exactement** l'Étape 1.1 |
| backlog | `b → ∞` | le backlog converge vers le comportement lost-sales |

In [ ]:
ext = pd.DataFrame(results["extensions"])
display(ext)

**Limite de validation, à reprendre telle quelle dans le rapport.**

Le périmètre du projet dépasse celui du papier de référence sur trois points, et ces
trois points ne sont **pas** validés au même niveau que le cœur du modèle :

- `N > 2` : les opérateurs `T0`, `T2`, `Tλ` sont génériques en `N` et le papier les
  écrit ainsi, mais la Condition 1 (monotonicité des base-stock) n'est automatique que
  pour un processus de naissance-mort. Au-delà, elle est vérifiée explicitement sur la
  matrice de transition (`Problem.satisfies_condition_1`) ;
- `K > 0` : le papier suppose explicitement « no set-up cost ». Le modèle retenu ici —
  coût de redémarrage du serveur, état augmenté `σ` — a été **redérivé**, et sa politique
  optimale est une bande d'hystérésis, non un `(s, S)` au sens de Chen & Simchi-Levi ;
- backlog : le papier suppose explicitement « lost sales ». L'extension à `x ∈ ℤ` avec
  coût de pénurie a également été redérivée.

Aucune Table ni Figure du papier ne peut servir de référence à ces trois volets.